In [1]:
pip install Boto3

   ---------------------------------------- 0.0/14.9 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/14.9 MB 13.0 MB/s eta 0:00:02
   -------------------- ------------------- 7.6/14.9 MB 24.2 MB/s eta 0:00:01
   ---------------------------------------- 14.9/14.9 MB 31.5 MB/s  0:00:00
  Attempting uninstall: botocore
    Found existing installation: botocore 1.40.49
    Uninstalling botocore-1.40.49:
      Successfully uninstalled botocore-1.40.49
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.0 requires botocore<1.40.50,>=1.40.46, but you have botocore 1.42.90 which is incompatible.


In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import io
import boto3
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError
from dotenv import load_dotenv

# 1. Chargement des variables d'environnement
load_dotenv()

# Configuration AWS (S3)
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "eu-west-3")
BUCKET_NAME = "projet-accidents-jedha"

# Configuration Base de données (RDS)
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME")

# 2. Initialisation des clients
# Client S3 pour la lecture
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# Engine SQLAlchemy pour l'écriture RDS
connection_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

# 3. Liste des fichiers à traiter (S3_Key : Nom_Table_SQL)
fichiers_a_pousser = {
    'silver/caracteristiques_cleaned.csv': 'fact_caracteristiques',
    'silver/usagers_cleaned.csv': 'dim_usagers',
    'silver/vehicules_cleaned.csv': 'dim_vehicules',
    'silver/lieux_cleaned.csv': 'fact_lieux'
}

def run_pipeline():
    print(f"🚀 Début de l'automatisation vers l'hôte : {DB_HOST}")
    
    try:
        # Test de la connexion RDS avant de commencer
        with engine.connect() as conn:
            print("✅ Connexion à la base de données validée.")

        for s3_key, table_name in fichiers_a_pousser.items():
            print(f"---")
            try:
                print(f"⏳ Récupération de {s3_key} sur S3...")
                response = s3_client.get_object(Bucket=BUCKET_NAME, Key=s3_key)
                
                # On utilise le moteur standard 'c' si possible pour la rapidité
                df = pd.read_csv(io.BytesIO(response['Body'].read()),sep=';', low_memory=False)
                print(f"✅ Données récupérées ({len(df)} lignes).")
                
                print(f"📤 Injection dans RDS : {table_name} (par paquets de 10k)...")
                # chunksize=10000 : On envoie par blocs pour éviter de saturer la mémoire/réseau
                df.to_sql(table_name, engine, if_exists='replace', index=False, method='multi', chunksize=10000)
                
                print(f"⭐ Succès pour la table {table_name} !")

            except Exception as e:
                print(f"❌ Erreur sur {table_name} : {e}")
                # LA SOLUTION AU PROBLÈME : on annule la transaction ratée pour débloquer la suite
                print("🔄 Nettoyage de la connexion (Rollback)...")
                with engine.connect() as conn:
                    # SQLAlchemy 2.0 gère cela automatiquement en sortant du bloc, 
                    # mais forcer une nouvelle connexion "propre" aide souvent.
                    pass 
                continue # On passe au fichier suivant

        print(f"\n✨ Félicitations ! Ton entrepôt de données est à jour.")

    except SQLAlchemyError as e:
        print(f"❌ Erreur SQL : {e}")
    except Exception as e:
        print(f"❌ Une erreur inattendue est survenue : {e}")

if __name__ == "__main__":
    run_pipeline()

AttributeError: partially initialized module 'pandas' from 'c:\Users\cuadr\anaconda3\Lib\site-packages\pandas\__init__.py' has no attribute 'core' (most likely due to a circular import)